# 02 - Silver Transform Incremental

**Purpose of the Silver Layer:** This notebook reads new Bronze snapshots, parses the raw JSON payloads, cleans and types the fields, and merges the result into the Silver table. The Silver layer is the cleaned and structured version of the raw Bronze data. 

So far in this project, the Bronze layer stores the raw BikePoint API response as JSON. That is useful for audit and reprocessing, however it is not ideal for quering data directly. The Silver layer converts that raw JSON into a clean structured table with proper columns, correct data types and useful derived features.

**Source table:** `workspace.bikepoint.bronze_bikepoint_raw`  
**Target table:** `workspace.bikepoint.silver_bikepoint`

**Grain:** The Silver table follows the same grain as Bronze:

- **One row per BikePoint station per pipeline snapshot.**
- This means each row represents the state of one BikePoint station at one API ingestion time.
- Silver does not aggregate the data. It only parses, cleans, types, and adds row-level calculations.

**Key Design Choice: Incremental Processing:** 
- One important design decision was to use **incremental processing** instead of reprocessing the full Bronze table every time.
- The Bronze table grows over time because it stores repeated API snapshots. If we processed the full Bronze table on every run, the pipeline would become slower as more history is collected.
- Instead of rebuilding the full Silver table every time, it checks which Bronze records have not already been processed into Silver.

The natural key is:

`bikepoint_id + pipeline_run_id`

Only rows that are new are transformed and written into Silver.

This is useful because:
- It avoids repeating work
- It reduces compute cost
- It makes the pipeline faster as history grows
- It supports scheduled runs every few hours
- It handles re-runs more safely

**To DO:**
- Check whether the Silver table already exists.
- Read the Bronze table.
- If Silver exists, anti-join Bronze against Silver keys to find only new records.
- If Silver does not exist, process all Bronze records.
- Remove technical duplicates using `bikepoint_id + pipeline_run_id`.
- Parse the raw JSON payload.
- Extract station fields and bike/dock values from `additionalProperties`.
- Add simple row-level calculations.
- Merge the processed batch into the Silver table.

**Why Anti-Join:**
A `LEFT ANTI JOIN` returns rows from Bronze that do not already exist in Silver. This prevents the same Bronze record from being processed again. The anti-join is based on:

`bikepoint_id + pipeline_run_id`

**Merge:** The final write uses Delta `MERGE`.

This makes the write safer than a simple append because the merge key prevents duplicate Silver rows. If we ever intentionally reprocess records, the merge can update existing rows instead of creating duplicates.

**What Silver does:**
- Reads raw Bronze records
- Removes technical duplicates
- Parses JSON from `raw_payload`
- Extracts BikePoint station details
- Extracts bike and dock counts from `additionalProperties`
- Converts strings into proper types
- Adds simple arithmetic fields such as:
  - `occupancy_rate`
  - `empty_rate`
  - `is_empty`
  - `is_full`
  - `broken_dock_count`
  - `is_consistent`

In [0]:
# Stores table names and project settings in one place.
CATALOG = "workspace"
SCHEMA = "bikepoint"

BRONZE_TABLE = "bronze_bikepoint_raw"
SILVER_TABLE = "silver_bikepoint"

BRONZE_TABLE_FQN = f"{CATALOG}.{SCHEMA}.{BRONZE_TABLE}"
SILVER_TABLE_FQN = f"{CATALOG}.{SCHEMA}.{SILVER_TABLE}"

print(f"Source table: {BRONZE_TABLE_FQN}")
print(f"Target table: {SILVER_TABLE_FQN}")

## Check Required Tables

Before transforming data, the notebook checks whether the Bronze table exists. The notebook also checks whether the Silver table already exists.

- If Silver does not exist, this is the first run and all Bronze data will be processed.
- If Silver exists, only new Bronze records will be processed.

In [0]:
# Bronze must exist before this notebook runs.
# Silver may or may not exist, depending on whether this is the first run.
from pyspark.sql import functions as func

# Create schema just in case it does not already exist.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

def table_exists(catalog_name, schema_name, table_name):
    result = spark.sql(
        f"SHOW TABLES IN {catalog_name}.{schema_name} LIKE '{table_name}'"
    )
    return result.count() > 0


bronze_exists = table_exists(CATALOG, SCHEMA, BRONZE_TABLE)
silver_exists = table_exists(CATALOG, SCHEMA, SILVER_TABLE)

if not bronze_exists:
    raise Exception(
        f"Bronze table does not exist: {BRONZE_TABLE_FQN}. "
        "Run 01_bronze_ingestion first."
    )

if silver_exists:
    print(f"Bronze table found: {BRONZE_TABLE_FQN}")
    print(f"Silver table found: {SILVER_TABLE_FQN}")
    print("This will be an incremental Silver run.")
else:
    print(f"Bronze table found: {BRONZE_TABLE_FQN}")
    print("Silver table does not exist yet.")
    print("This will be the first Silver run.")

## Read Bronze and Existing Silver Keys

This step reads all Bronze records. If the Silver table already exists, the notebook also reads only the Silver key columns:

`bikepoint_id` and `pipeline_run_id`

These keys are used to find Bronze rows that have not yet been processed. We do not need to read all Silver columns for this step because the anti-join only needs the key columns.

In [0]:
# Read the full Bronze table.
bronze_df = spark.table(BRONZE_TABLE_FQN)
bronze_total_rows = bronze_df.count()
print(f"Bronze total rows: {bronze_total_rows:,}")

if silver_exists:
    silver_keys_df = (
        spark.table(SILVER_TABLE_FQN)
        .select("bikepoint_id", "pipeline_run_id")
        .distinct()
    )
    silver_key_count = silver_keys_df.count()
    print(f"Existing Silver keys: {silver_key_count:,}")
else:
    silver_keys_df = None
    silver_key_count = 0
    print("Silver Layer does not exist yet")

## Find New Bronze Records

This step finds the Bronze records that have not yet been processed into Silver.

- If Silver exists, we use a `LEFT ANTI JOIN`.
- If Silver does not exist yet, every Bronze row is treated as new.

In [0]:
if silver_exists:
    # LEFT ANTI JOIN means
    incremental_bronze_df = bronze_df.join( 
        silver_keys_df,
        on=["bikepoint_id", "pipeline_run_id"],
        how="left_anti" # keep rows from Bronze where the key does NOT exist in Silver
    )
else: # If first run, process all Bronze records.
    incremental_bronze_df = bronze_df

incremental_row_count = incremental_bronze_df.count()
skipped_row_count = bronze_total_rows - incremental_row_count

print(f"Bronze total rows:           {bronze_total_rows:,}")
print(f"Already processed rows:      {skipped_row_count:,}")
print(f"New rows to process:         {incremental_row_count:,}")

## Exit Early If There Is No New Data

If there are no new Bronze records to process, we should exit the notebook early. This is useful because otherwise the notebook will run multiple times. If Silver is already up to date, there is no need to parse or merge anything.

In [0]:
# In Databricks, dbutils.notebook.exit stops the notebook cleanly.
print(incremental_row_count)
if incremental_row_count == 0:
    print("Silver is already up to date. No new Bronze records to process.")
    dbutils.notebook.exit("no_new_data")

## Remove Technical Duplicates
 
Silver should contain only one row per:
 
`bikepoint_id + pipeline_run_id`

This step removes technical duplicates from the incremental Bronze batch. This does not remove valid repeated snapshots from different pipeline runs. This step only protects against accidental duplicates inside the same pipeline run.

In [0]:
# The natural key is bikepoint_id + pipeline_run_id.
rows_before_dedup = incremental_row_count
incremental_bronze_df = incremental_bronze_df.dropDuplicates(
    ["bikepoint_id", "pipeline_run_id"]
)

rows_after_dedup = incremental_bronze_df.count()
duplicates_removed = rows_before_dedup - rows_after_dedup

if duplicates_removed > 0:
    print(f"Removed {duplicates_removed:,} technical duplicate rows")
else:
    print(f"No technical duplicates found. Rows passing through: {rows_after_dedup:,}")

## Parse the Raw JSON Payload

The Bronze table stores the API response in a column called `raw_payload`, which is currently in the string format. In the Silver layer, we parse this JSON string into a structured Spark column. For this, we design the schema explicitly because it is safer and clearer than asking Spark to guess the structure.

**Before we start Parsing the `raw_payload` column, let's first better understand the type of data we are dealing with.**

In [0]:
import json

# Grab one row from the incremental batch
sample_row = incremental_bronze_df.select("raw_payload").limit(1).collect()[0]

# Parse the JSON string into a Python dict, then pretty-print it
sample_json = json.loads(sample_row["raw_payload"])
print(json.dumps(sample_json, indent=2))

In [0]:
# To better understand the data, let's have a look at the important fields that are present inside the JSON in the form of keys.
sample_json = json.loads(
    incremental_bronze_df.select("raw_payload").limit(1).collect()[0]["raw_payload"]
)

print("Top Level Attributes:")
for key in sample_json.keys():
    print(f". {key}: {type(sample_json[key]).__name__}")

print("\nKeys inside additionalProperties:")
for prop in sample_json.get("additionalProperties", []):
    # print(prop)
    print(f". {prop['key']}")

In [0]:
# Important values like NbBikes and NbDocks live inside additionalProperties.
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, ArrayType

# Schema for one item inside additionalProperties.
additional_property_schema = StructType([
    StructField("key", StringType(), nullable=True),
    StructField("value", StringType(), nullable=True),
    StructField("modified", StringType(), nullable=True),
])

# Schema for one BikePoint object.
station_schema = StructType([
    StructField("id", StringType(), nullable=True),
    StructField("commonName", StringType(), nullable=True),
    StructField("placeType", StringType(), nullable=True),
    StructField("lat", DoubleType(), nullable=True),
    StructField("lon", DoubleType(), nullable=True),
    StructField("additionalProperties", ArrayType(additional_property_schema), nullable=True),
])

# Right now, raw_payload is a string of text, Spark sees it as characters, not as structured data. 
# We need to turn the string into a navigable struct.
parsed_df = incremental_bronze_df.withColumn(
    # Name of the new column. After this runs, the DataFrame has all the
    # original Bronze columns PLUS a new "temp" column containing the
    # parsed struct.
    "temp",
    func.from_json(func.col("raw_payload"), station_schema) # func.from_json is the tool that turns the string into structured data. 
)

parsed_df.printSchema()

In [0]:
parsed_df.limit(5).display()

## Validate JSON Parsing

**`func.from_json` is the tool that turns the string into structured data.**

`from_json` can return null if a payload cannot be parsed. This check makes sure the JSON parsing worked before we continue. If parsing fails, the notebook raises an error instead of silently creating bad Silver data which is ideal in production.

In [0]:
# If temp is null, it means from_json failed for that row
# Check whether the JSON parsing worked correctly
# The column "temp" was created using from_json(raw_payload, schema).
# If from_json cannot parse a raw JSON string, it returns null for that row.
failed_parse_count = parsed_df.filter(func.col("temp").isNull()).count()

# If any rows failed to parse, and exception would be raised,
# which prevents bad or incomplete data from moving into the Silver layer.
if failed_parse_count > 0:
    raise Exception(f"JSON parsing failed for {failed_parse_count:,} rows.")

print("JSON parsing successful.")

## Extract temp Fields and BikePoint Properties

The API has two types of fields:

### 1. Top Level Fields:

Examples:

- `id`
- `commonName`
- `placeType`
- `lat`
- `lon`

### 2. Nested Additional Properties

Important operational values are stored inside the `additionalProperties` array.

Examples:

- `NbBikes`
- `NbEmptyDocks`
- `NbDocks`
- `NbStandardBikes`
- `NbEBikes`
- `Installed`
- `Locked`
- `Temporary`
- `TerminalName`

We use a helper function to extract these values by key.

In [0]:
# Extract fields and build the flat Silver DataFrame
def get_attribute(additional_props_col, key_name): 
    # Extracts the 'value' field from the additionalProperties array where the property key matches key_name.
    matching_items = func.filter(additional_props_col, lambda item: item["key"] == func.lit(key_name)) # func.lit(key_name) means treat this Python value as a literal Spark column.
    first_item = func.element_at(matching_items, 1)
    return first_item["value"]

additional_props = func.col("temp.additionalProperties")
parsed_df.select(additional_props).limit(5).display()

In [0]:

silver_df = parsed_df.select(
    # Identifiers and location, and are the Top-Level Fields. 
    func.col("bikepoint_id"),
    func.col("temp.commonName").alias("station_name"),
    func.col("temp.placeType").alias("place_type"),
    func.col("temp.lat").alias("latitude"),
    func.col("temp.lon").alias("longitude"),

    # Bike and dock counts located in the "additionalProperties" field
    # The API returns these as strings, so we cast them to integers using .cast()
    get_attribute(additional_props, "NbBikes").cast("int").alias("nb_bikes"),
    get_attribute(additional_props, "NbEmptyDocks").cast("int").alias("nb_empty_docks"),
    get_attribute(additional_props, "NbDocks").cast("int").alias("nb_docks"),
    get_attribute(additional_props, "NbStandardBikes").cast("int").alias("nb_standard_bikes"),
    get_attribute(additional_props, "NbEBikes").cast("int").alias("nb_ebikes"),

    # Status flags located in the "additionalProperties" field
    # The API returns true/false as strings, so we convert them to booleans.
    (get_attribute(additional_props, "Installed") == "true").alias("is_installed"),
    (get_attribute(additional_props, "Locked") == "true").alias("is_locked"),
    (get_attribute(additional_props, "Temporary") == "true").alias("is_temporary"),

    # Reference field located in the "additionalProperties" field
    get_attribute(additional_props, "TerminalName").alias("terminal_name"),

    # Metadata from the Bronze layer
    func.col("ingestion_ts"),
    func.col("pipeline_run_id"),
    func.col("source_url"),
    func.col("http_status"),
    func.col("station_count"),
)

display(silver_df.limit(5))

## Add Row Level Arrtibutes

This step adds simple arithmetic fields. These are allowed in Silver because they are direct calculations from the same row.

The Silver table contains cleaned and typed BikePoint station data. Some columns come directly from the TfL API, while others are derived from simple row-level calculations.


### 1. Source and Parsed Columns

| Column | Meaning |
|---|---|
| `nb_bikes` | Number of bikes currently available at that station. If `nb_bikes = 8`, users can hire 8 bikes from that station. |
| `nb_empty_docks` | Number of empty docking spaces currently available. If `nb_empty_docks = 12`, users can return up to 12 bikes at that station. |
| `nb_docks` | Total number of docking spaces at the station. If `nb_docks = 25`, the station has 25 dock spaces in total. |
| `nb_ebikes` | Number of electric bikes currently available at the station. If `nb_ebikes = 3`, there are 3 e-bikes available. |
| `ingestion_ts` | The UTC timestamp when our pipeline collected the API data. This tells us when the snapshot was captured. |


### 2. Derived Columns

| Column | Meaning |
|---|---|
| `occupancy_rate` | Shows how full the station is with bikes. It is calculated as `nb_bikes / nb_docks`. |
| `empty_rate` | Shows how much free return space the station has. It is calculated as `nb_empty_docks / nb_docks`. |
| `total_accounted_docks` | Total number of docks accounted for by either bikes or empty docks. It is calculated as `nb_bikes + nb_empty_docks`. |
| `broken_dock_count` | Number of docks not accounted for by bikes or empty docks. These may be broken, locked, unavailable, or not reporting correctly. |
| `is_empty` | Boolean flag showing whether the station has no bikes available. `true` means users cannot hire a bike from this station. |
| `is_full` | Boolean flag showing whether the station has no empty docks available. `true` means users cannot return a bike to this station. |
| `is_consistent` | Data quality flag showing whether the bike and dock counts make logical sense. It checks that `nb_bikes + nb_empty_docks <= nb_docks`. |
| `ebike_share` | Percentage of available bikes that are e-bikes. It is calculated as `nb_ebikes / nb_bikes`. |
| `ingestion_ts_local` | The ingestion timestamp converted from UTC to London local time. |
| `service_date` | The local London date of the snapshot. |
| `service_hour` | The local London hour of the snapshot. This helps with time-of-day analysis. |
| `day_of_week` | Numeric day of the week from the local timestamp. In Spark, Sunday is `1` and Saturday is `7`. |
| `is_weekend` | Boolean flag showing whether the snapshot was taken on a Saturday or Sunday. |

These columns make the Silver table easier to analyse in the Gold layer. The Gold layer can use them to calculate station stress, availability patterns, and investment priority.

In [0]:
silver_df = (
    silver_df
    .withColumn("occupancy_rate", func.when(func.col("nb_docks") > 0, func.col("nb_bikes") / func.col("nb_docks")))
    .withColumn("empty_rate", func.when(func.col("nb_docks") > 0, func.col("nb_empty_docks") / func.col("nb_docks")))
    .withColumn("total_active_docks", func.col("nb_bikes") + func.col("nb_empty_docks"))
    .withColumn("broken_dock_count", func.greatest(
            func.col("nb_docks") - func.col("total_active_docks"),
            func.lit(0)
        )
    )
    
    # Station level availability flags
    .withColumn("is_empty", func.col("nb_bikes") == 0)
    .withColumn("is_full",  func.col("nb_empty_docks") == 0)
    .withColumn("is_consistent", func.col("total_active_docks") <= func.col("nb_docks"))

    .withColumn("ebike_share", func.when(func.col("nb_bikes") > 0, func.col("nb_ebikes") / func.col("nb_bikes")))
    
    .withColumn("ingestion_ts_local", func.from_utc_timestamp(func.col("ingestion_ts"), "Europe/London"))
    .withColumn("service_date",  func.to_date(func.col("ingestion_ts_local")))
    .withColumn("service_hour",  func.hour(func.col("ingestion_ts_local")))
    .withColumn("day_of_week",   func.dayofweek(func.col("ingestion_ts_local")))
    .withColumn("is_weekend",    func.col("day_of_week").isin(1, 7))
)

silver_df.printSchema()

In [0]:
silver_df.limit(5).display()

## Merge into the Silver Delta Table

There are two paths:

### First Run:

If the Silver table does not exist, the notebook creates it from the processed batch from the Bronze table.

### Incremental runs

If the Silver table already exists, the notebook uses Delta `MERGE`. The merge condition is:

`target.bikepoint_id = source.bikepoint_id`
 and 
`target.pipeline_run_id = source.pipeline_run_id`

This protects Silver from duplicate rows. In normal runs, the anti-join already sends only new records, so the merge usually inserts new rows. 

In [0]:

# Count how many rows are in this Silver batch.
batch_size = silver_df.count()

if not silver_exists:
    # First time running Silver will create the table from the full batch.
    (
        silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(SILVER_TABLE_FQN)
    )
    rows_inserted = batch_size
    rows_updated = 0
else:
    # Silver layer already exists, so merge this new batch into it.
    # Make the DataFrame available to SQL for the MERGE statement.
    silver_df.createOrReplaceTempView("silver_incremental_batch") # SQL cannot directly read a Python DataFrame variable called silver_df for the Merge query.
    rows_before_merge = spark.table(SILVER_TABLE_FQN).count()
    merge_sql = f"""
        MERGE INTO {SILVER_TABLE_FQN} AS target
        USING silver_incremental_batch AS source
        ON  target.bikepoint_id = source.bikepoint_id
        AND target.pipeline_run_id = source.pipeline_run_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """
    spark.sql(merge_sql)

    rows_after_merge = spark.table(SILVER_TABLE_FQN).count()
    rows_inserted = rows_after_merge - rows_before_merge
    rows_updated = batch_size - rows_inserted

print(f"Batch size:    {batch_size:,}")
print(f"Rows inserted: {rows_inserted:,}")
print(f"Rows updated:  {rows_updated:,}")

## Verify the Silver Table

This final step checks that the Silver table was written correctly.

The verification query checks:
- total rows
- unique stations
- number of snapshots
- earliest and latest pipeline run
- empty station observations
- full station observations
- broken dock count
- average occupancy rate
- inconsistent rows

The most important quality check is `inconsistent_rows`. This should ideally be zero because bike count plus empty dock count should not exceed total dock capacity.

In [0]:
verification_df = spark.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT bikepoint_id) AS unique_stations,
        COUNT(DISTINCT pipeline_run_id) AS distinct_snapshots,
        MIN(pipeline_run_id) AS earliest_pipeline_run,
        MAX(pipeline_run_id) AS latest_pipeline_run,
        MIN(ingestion_ts) AS earliest_ingestion_ts,
        MAX(ingestion_ts) AS latest_ingestion_ts,
        SUM(CAST(is_empty AS INT)) AS empty_station_observations,
        SUM(CAST(is_full AS INT)) AS full_station_observations,
        SUM(broken_dock_count) AS total_broken_dock_count,
        ROUND(AVG(occupancy_rate), 4) AS avg_occupancy_rate,
        ROUND(AVG(ebike_share), 4) AS avg_ebike_share
    FROM {SILVER_TABLE_FQN}
""")

verification_df.show(truncate=False, vertical=True)